In [1]:
# ─────────────────────────────────────────
# os     : 환경변수를 읽기 위한 모듈
# openai : OpenAI API를 호출하기 위한 모듈
# dotenv : .env 파일에서 API 키를 불러오기 위한 모듈
# ─────────────────────────────────────────
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

# ─────────────────────────────────────────
# API 연결 테스트
# client.chat.completions.create() : ChatGPT API 호출
#
# model            : 사용할 모델 이름
# messages         : 대화 이력 (role + content 딕셔너리 리스트)
#   role='system'  : 에이전트 역할/규칙 정의
#   role='user'    : 사용자 입력
#   role='assistant: LLM 응답 (대화 이력에 추가할 때 사용)
# temperature=0    : 응답의 무작위성 제어
#                    0에 가까울수록 일관된 답변
#                    ReAct에서는 0으로 설정하는 게 유리
#                    → 매번 같은 형식으로 출력해야 파싱이 안정적
# max_completion_tokens : 응답 최대 토큰 수 제한
# ─────────────────────────────────────────
response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[
        {'role': 'user', 'content': 'ReAct agent에 대해서 한줄로 설명해줘'}
    ],
    temperature=0,
    max_completion_tokens=100
)

# ─────────────────────────────────────────
# 응답 텍스트 추출
# response.choices      : 응답 후보 리스트 (보통 1개)
# [0]                   : 첫번째 응답
# .message.content      : 실제 텍스트 내용
# ─────────────────────────────────────────
print(response.choices[0].message.content)

# ─────────────────────────────────────────
# 토큰 사용량 확인
# prompt_tokens     : 입력에 사용된 토큰 수
# completion_tokens : 출력에 사용된 토큰 수
# total_tokens      : 전체 토큰 수 (과금 기준)
# ─────────────────────────────────────────
print(f"토큰 사용량: {response.usage}")
print(f"총 토큰: {response.usage.total_tokens}")

ReAct agent는 외부 환경과 상호작용하며 반응하고 학습하는 인공지능 시스템으로, 주어진 상황에 따라 적절한 행동을 선택하는 능력을 갖추고 있습니다.
토큰 사용량: CompletionUsage(completion_tokens=44, prompt_tokens=18, total_tokens=62, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))
총 토큰: 62


In [2]:
import re  # Action 파싱할 때 정규표현식 사용

# ─────────────────────────────────────────
# SimpleReActAgent : 1번, 2번 파일에서 만든 구조를
# 실제 LLM과 연동한 완성된 ReAct 에이전트 클래스
# ─────────────────────────────────────────
class SimpleReActAgent:

    # ─────────────────────────────────────────
    # model     : 사용할 OpenAI 모델 이름
    # max_steps : 무한루프 방지용 최대 반복 횟수
    # tools     : Tool을 저장하는 딕셔너리
    #             key   = Tool 이름
    #             value = {'func': 실행함수, 'description': 설명}
    # system_prompt : 매 run()마다 새로 생성되는 시스템 프롬프트
    # ─────────────────────────────────────────
    def __init__(self, model='gpt-4o-mini', max_steps=5):
        self.model = model
        self.max_steps = max_steps
        self.tools = {}             # 빈 딕셔너리로 시작
        self.system_prompt = ''     # 나중에 _build_system_prompt()로 채워짐

    # ─────────────────────────────────────────
    # Tool을 등록하는 메서드
    # 2번 파일의 ToolRegistry.register()와 같은 역할
    # func        : 실제 실행할 파이썬 함수
    # description : LLM이 언제 이 Tool을 쓸지 판단하는 근거
    # ─────────────────────────────────────────
    def register_tool(self, name, func, description):
        self.tools[name] = {
            'func': func,
            'description': description
        }

    # ─────────────────────────────────────────
    # 시스템 프롬프트를 자동으로 생성하는 메서드
    # 등록된 Tool 목록을 프롬프트에 자동으로 포함시킴
    # run()이 호출될 때 실행되어 self.system_prompt에 저장
    # ─────────────────────────────────────────
    def _build_system_prompt(self):
        # ─────────────────────────────────────────
        # 등록된 모든 Tool의 이름과 설명을 한 줄씩 정리
        # join() : 리스트를 줄바꿈으로 연결해서 하나의 문자열로
        # ─────────────────────────────────────────
        tool_desc = '\n'.join([
            f"- {name}: {info['description']}"
            for name, info in self.tools.items()
        ])

        # ─────────────────────────────────────────
        # 4단계 ReActPromptTemplate.build()와 같은 역할
        # 시스템 지시문 + Tool 목록 + 출력 형식을 하나로 조립
        # LLM이 이 형식을 따라야 파서가 안정적으로 동작
        # ─────────────────────────────────────────
        self.system_prompt = f'''당신은 주어진 질문에 정확하게 답변하기 위해 도구를 사용하는 AI 에이전트입니다.
        사용 가능한 도구:
        {tool_desc}

        반드시 다음 형식을 따라 응답하세요:
        Thought: [현재 상황에 대한 추론]
        Action: [도구이름][입력값]

        도구 실행결과는 Observation으로 제공됩니다
        최종 답변을 제출할 때는 반드시 다음 형식을 사용하세요:
        Thought: [최종 추론]
        Action: Finish[최종 답변]

        중요: 한 번에 하나의 Action만 수행하세요
        '''

    # ─────────────────────────────────────────
    # LLM 출력에서 Action 이름과 입력값을 추출하는 메서드
    # 1번 파일의 ReActParser.extract_action()과 같은 역할
    # 반환값 : (action_name, action_input) 튜플
    #          파싱 실패시 (None, None) 반환
    # ─────────────────────────────────────────
    def _parse_action(self, text):
        match = re.search(r'Action\s*:\s*(\w+)\[(.+?)\]', text, re.DOTALL)
        if match:
            return match.group(1).strip(), match.group(2).strip()
        return None, None  # 파싱 실패시 None 반환

    # ─────────────────────────────────────────
    # Tool을 실행하는 메서드
    # 2번 파일의 ToolRegistry.execute()와 같은 역할
    #
    # Finish는 Tool이 아니라 종료 신호
    # → None을 반환해서 run()에서 루프를 종료하도록
    # ─────────────────────────────────────────
    def _execute_tool(self, tool_name, tool_input):
        if tool_name == 'Finish':
            return None  # 종료 신호 → run()에서 처리

        elif tool_name in self.tools:
            try:
                # ─────────────────────────────────────────
                # self.tools[tool_name]['func'] : 등록된 실행 함수
                # tool_input을 인자로 넘겨서 실행
                # ─────────────────────────────────────────
                return self.tools[tool_name]['func'](tool_input)
            except Exception as e:
                return f'도구 실행 오류: {e}'

        else:
            return f"'{tool_name}' 도구를 찾을 수 없습니다."

    # ─────────────────────────────────────────
    # 실제 ReAct 루프를 실행하는 메서드
    # 1번+2번 파일의 simulate_react_with_tools()와 같은 역할
    # 단, 이번엔 LLM이 직접 Thought와 Action을 생성
    # ─────────────────────────────────────────
    def run(self, question):

        # 시스템 프롬프트 생성 (등록된 Tool 목록 반영)
        self._build_system_prompt()

        # ─────────────────────────────────────────
        # messages : 대화 이력을 쌓는 리스트
        # system   : 에이전트 규칙 (루프 내내 고정)
        # user     : 처음 질문
        # 루프를 돌면서 assistant/user 메시지가 계속 추가됨
        # → LLM이 이전 대화 전체를 보고 다음 Thought 생성
        # ─────────────────────────────────────────
        messages = [
            {'role': 'system', 'content': self.system_prompt},
            {'role': 'user',   'content': f'Question: {question}'}
        ]

        print(f'Question: {question}')
        print('=' * 60)

        # ─────────────────────────────────────────
        # max_steps만큼 루프 반복
        # Finish가 나오면 중간에 return으로 종료
        # max_steps까지 Finish 없으면 루프 자연 종료
        # ─────────────────────────────────────────
        for step in range(self.max_steps):

            # ─────────────────────────────────────────
            # LLM 호출 → Thought + Action 생성
            # messages 전체를 넘겨서 이전 맥락 유지
            # ─────────────────────────────────────────
            response = client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=0
            )

            # LLM이 생성한 텍스트 추출
            assistant_msg = response.choices[0].message.content

            # ─────────────────────────────────────────
            # LLM 응답을 대화 이력에 추가
            # 다음 루프에서 LLM이 이 응답을 참고해서
            # 다음 Thought를 생성할 수 있도록
            # ─────────────────────────────────────────
            messages.append({
                'role': 'assistant',
                'content': assistant_msg
            })

            print(f'\n-- Step {step + 1} --')
            print(assistant_msg)

            # LLM 출력에서 Action 파싱
            action_name, action_input = self._parse_action(assistant_msg)

            # ─────────────────────────────────────────
            # Case 1 : Finish → 루프 종료
            # ─────────────────────────────────────────
            if action_name == 'Finish':
                print(f"\n{'=' * 60}")
                print(f'Final Answer: {action_input}')
                return action_input

            # ─────────────────────────────────────────
            # Case 2 : 일반 Tool → 실행 후 Observation을
            #          user 메시지로 추가
            #          LLM이 다음 루프에서 이 결과를 보고
            #          다음 Thought를 생성
            # ─────────────────────────────────────────
            elif action_name:
                observation = self._execute_tool(action_name, action_input)
                obs_msg = f'Observation: {observation}'
                messages.append({
                    'role': 'user',     # Observation은 user 역할로 추가
                    'content': obs_msg
                })
                print(f'\n{obs_msg}')

            # ─────────────────────────────────────────
            # Case 3 : 파싱 실패 → 형식 오류 메시지를
            #          user 메시지로 추가해서 LLM이 수정하도록
            # ─────────────────────────────────────────
            else:
                messages.append({
                    'role': 'user',
                    'content': '형식 오류: Action: 도구이름[입력값] 형식으로 응답해주세요'
                })

        # max_steps 도달시
        print('\n최대 단계수에 도달했습니다.')
        return None

In [3]:
# ─────────────────────────────────────────
# calculator : 수식 계산 Tool
# 2번 파일의 CalculatorTool.execute()와 같은 역할
# 이번엔 클래스 없이 순수 함수로 구현
# ─────────────────────────────────────────
def calculator(expression):
    # ─────────────────────────────────────────
    # 허용된 문자만 통과시켜서 eval() 보안 처리
    # 2번 파일과 동일한 보안 로직
    # ─────────────────────────────────────────
    allowed = set('0123456789+-*/.() ')
    if all(c in allowed for c in expression):
        return str(eval(expression))  # 결과를 문자열로 반환
    return "허용되지 않는 수식입니다."


# ─────────────────────────────────────────
# knowledge_base : 내장 지식베이스 검색 Tool
# 2번 파일의 DictionaryTool.execute()와 같은 역할
# ─────────────────────────────────────────
def knowledge_base(query):
    # ─────────────────────────────────────────
    # 실제 서비스에서는 DB나 외부 API로 대체될 부분
    # 지금은 하드코딩된 dict로 구현
    # ─────────────────────────────────────────
    kb = {
        "서울 인구": "서울특별시의 인구는 약 950만 명입니다 (2024년 기준).",
        "서울 면적": "서울특별시의 면적은 약 605.2 km2 입니다.",
        "한국 GDP": "대한민국의 GDP는 약 1조 7천억 달러입니다 (2024년 기준).",
        "한국 인구": "대한민국의 총 인구는 약 5,170만 명입니다 (2024년 기준).",
        "파이썬":   "Python은 1991년 귀도 반 로섬이 만든 프로그래밍 언어입니다.",
        "인공지능": "인공지능(AI)은 인간의 지능을 모방하는 컴퓨터 시스템을 연구하는 분야입니다."
    }

    # ─────────────────────────────────────────
    # 대소문자 구분 없이 검색하기 위해 소문자로 변환
    # query_lower : "서울 인구" → "서울 인구" (한글은 변화 없음)
    # ─────────────────────────────────────────
    query_lower = query.lower()

    for key, value in kb.items():
        # ─────────────────────────────────────────
        # 검색 조건 1 : key가 query에 포함되는 경우
        #              예) query="서울 인구 알려줘", key="서울 인구" → 매칭
        # 검색 조건 2 : key의 모든 단어가 query에 포함되는 경우
        #              예) query="서울의 면적은?", key="서울 면적"
        #              → "서울" in query AND "면적" in query → 매칭
        # ─────────────────────────────────────────
        if key in query_lower or all(word in query_lower for word in key.split()):
            return value

    return f"'{query}'에 대한 정보를 찾을 수 없습니다."


# ─────────────────────────────────────────
# Agent 생성 + Tool 등록 + 실행
# ─────────────────────────────────────────

# max_steps=10 : 최대 10번까지 루프 허용
agent = SimpleReActAgent(max_steps=10)

# ─────────────────────────────────────────
# Tool 등록
# register_tool(이름, 실행함수, 설명)
# 이름은 LLM이 Action에서 쓸 이름
# → 프롬프트에 자동으로 포함되어 LLM이 참고
# ─────────────────────────────────────────
agent.register_tool(
    'Calc',
    calculator,
    '수학 수식을 계산합니다. 예) Calc[2+3]'
)
agent.register_tool(
    'knowbs',
    knowledge_base,
    '내장 지식베이스에서 정보를 검색합니다. 예) knowbs[서울 인구]'
)

# ─────────────────────────────────────────
# Agent 실행
# run() 내부에서 이런 순서로 동작
# 1. _build_system_prompt() → 프롬프트 생성
# 2. LLM 호출 → Thought + Action 생성
# 3. _parse_action() → Action 파싱
# 4. _execute_tool() → Tool 실행
# 5. Observation을 messages에 추가
# 6. Finish 나올 때까지 2~5 반복
# ─────────────────────────────────────────
result = agent.run('서울의 인구와 면적을 조회하고, 인구밀도(인구/면적)을 계산해주세요')

Question: 서울의 인구와 면적을 조회하고, 인구밀도(인구/면적)을 계산해주세요

-- Step 1 --
Thought: 서울의 인구와 면적을 먼저 조회한 후, 인구밀도를 계산해야 합니다. 인구와 면적 정보를 각각 확인하겠습니다. 
Action: knowbs[서울 인구]

Observation: 서울특별시의 인구는 약 950만 명입니다 (2024년 기준).

-- Step 2 --
Thought: 서울의 인구는 약 950만 명으로 확인되었습니다. 다음으로 서울의 면적을 조회해야 합니다. 
Action: knowbs[서울 면적]

Observation: 서울특별시의 면적은 약 605.2 km2 입니다.

-- Step 3 --
Thought: 서울의 인구는 약 950만 명, 면적은 약 605.2 km²로 확인되었습니다. 이제 인구밀도를 계산할 수 있습니다. 인구밀도는 인구를 면적으로 나누어 계산합니다. 
Action: Calc[9500000 / 605.2]

Observation: 15697.290152015861

-- Step 4 --
Thought: 서울의 인구밀도는 약 15697.29명/km²로 계산되었습니다. 
Action: Finish[서울의 인구밀도는 약 15697.29명/km²입니다.]

Final Answer: 서울의 인구밀도는 약 15697.29명/km²입니다.
